# FastQuote - Google Colab Demo

1. Upload this notebook to Google Colab.
2. Open **Secrets** in the left sidebar, add `OPENROUTER_API_KEY`, and enable notebook access.
3. Choose **Runtime > Run all**.
4. Use the FastQuote page displayed in the last cell.

The notebook reads the key through `google.colab.userdata`. It never asks for the key in a text field or stores it in quote data. The page is available only in your connected Colab session.

In [ ]:
import base64
import io
import shutil
import zipfile
from pathlib import Path

PROJECT_ZIP_B64 = '...'
PROJECT_ROOT = Path("/content/FastQuote")
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

archive_bytes = base64.b64decode(PROJECT_ZIP_B64)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    archive.extractall(PROJECT_ROOT)

print(f"FastQuote project prepared at {PROJECT_ROOT}")


In [ ]:
import os

SECRET_NAME = "OPENROUTER_API_KEY"
DEFAULT_MODEL = "openai/gpt-4.1-mini"

try:
    from google.colab import userdata
    api_key = (userdata.get(SECRET_NAME) or "").strip()
except Exception:
    api_key = os.environ.get(SECRET_NAME, "").strip()

if not api_key:
    raise RuntimeError(
        "OpenRouter API key was not found. In the Colab left sidebar, open "
        "Secrets, add OPENROUTER_API_KEY, and enable Notebook access."
    )

model = os.environ.get("OPENROUTER_MODEL", DEFAULT_MODEL).strip() or DEFAULT_MODEL

os.environ["OPENROUTER_API_KEY"] = api_key
os.environ["OPENROUTER_MODEL"] = model
print(f"OpenRouter Secret loaded successfully. Model: {model}")


In [ ]:
import json
import sys
import threading
import urllib.request
from http.server import ThreadingHTTPServer

sys.path.insert(0, str(PROJECT_ROOT))
from google.colab import output
from run_app import Handler

previous_server = globals().get("FASTQUOTE_SERVER")
if previous_server is not None:
    previous_server.shutdown()
    previous_server.server_close()

FASTQUOTE_SERVER = ThreadingHTTPServer(("127.0.0.1", 0), Handler)
FASTQUOTE_PORT = FASTQUOTE_SERVER.server_address[1]
FASTQUOTE_THREAD = threading.Thread(
    target=FASTQUOTE_SERVER.serve_forever, daemon=True
)
FASTQUOTE_THREAD.start()

base_url = f"http://127.0.0.1:{FASTQUOTE_PORT}"
with urllib.request.urlopen(base_url + "/api/health", timeout=10) as response:
    health = json.load(response)
sample_request = urllib.request.Request(
    base_url + "/api/quote",
    data=json.dumps({
        "request": "Please quote M16x50, 2 cartons.",
        "mode": "rules",
    }).encode("utf-8"),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(sample_request, timeout=10) as response:
    sample = json.load(response)
if health.get("status") != "ok" or sample.get("processing_status") != "quoted":
    raise RuntimeError("FastQuote server did not pass its startup check.")

print(f"FastQuote is ready: {health['catalog_rows']} catalog rows; sample quote passed.")
print("The page below works while this Colab runtime remains connected.")
output.serve_kernel_port_as_iframe(FASTQUOTE_PORT, height="800")
